INTRODUZIONE AL VAE: PROBABILITA' E INFERENZA VARIAZIONALE.

Passiamo dalla geometria deterministica alla proabilità
Sappiamo come l'autoencoder classico sia un eccellente archivista, capace di comprimere i dati ma privo di immaginazione.
Vediamo il varietional auto encoder (VAE), non impariamo solo a copiare la realtà ma a modellarne l'incertezza.
Immagina di non dover più solo riprodurre un volto, ma di possedere istruzioni per crearne infiniti nuovi.
Per fare questo salto dobbiamo abbandonare la sicurezza dei punti fissi per l'approccio bayesiano.

- Approccio Bayesiano: modellare l'incertezza nello spazio latente
- Reparameterization Trick: rendere derivabile il campionamento, soluzione tecnica che ha reso possibile addestrare questi modelli.
- Inferenza approssimata: gestire matematicamente l'impossibilità di calcolare ogni singola probabilità del mondo reale.

- Con autoencoder classico: x -> Encoder -> z -> Decoder ->circa x
- Con VAE: x -> Encoder -> distribuzione probabilistica di z -> campinamento di z -> decoder -> circa x
Qui, nel VAE il latent space (z) non è un insieme di punti arbitrari, ma uno spazio probabilistico regolarizzato.

Perchè le coordinate fisse sono un limite per la creatività?

L'approccio Bayesiano
Dalle coordinate fisse alla densità di probabilità
Nell'autoencoder classico (AE) ogni input (x) viene mappato in un punto statico. 
L'AE deve consegnare un pacco in un indirizzo specifico, se si sbagli indirizzo, la consegna fallisce.
Il VAE invece l'encoder non ci da più un indirizzo, ma una zona, una nuvola di probabilità.
Non so dove è esattamente l'indirizzo dove devi consegnare il pacco, ma sono sicura al 95% che si trovi in questo quartiere.
Il VAE cambia visione adottando un paradigma bayesiano.
Invece di produrre un singolo codice, l'encoder stima i parametri di una distribuzione di probabilità che descrive dove il dato potrebbe trovarsi.
Questo cambiamento trasforma la compressione da un atto meccanico, ad un dato statistico.

Come si traduce questa nuvola in termini di parametri latenti?

Distribuzione Latenti
Mappare l'incertezza.
-L'encoder non restituisce più un vettore statico 'z' ma due vettori la media e la varianza. Invece di un punto isolato, lo spazio  latente diventa una collezione di gaussiane. Lo spazio latente diventa un mare di nuvole, dove ogni nuvola rappresenta una categoria dei dati.
Modellare l'incertezza significa che un volto possa avere diverse varianti.
Questo permette di modellare esplicitamente l'incertezza nella compressione dei dati.
L'inferenza diventa quindi il compito di stimare la distribuzione condizionata dei codici latenti dato l'input osservato.
E' la fine del determinismo rigido.

Ma perchè vogliamo forzare questa stocasticità all'interno della nostra architettura?

Stocasticità Latente
Introdurre la stocasticità è come aggiungere una regolarizzazione naturale. Imponiamo una conoscenza a priori, ovvero chiediamo alla rete di fare in modo che le sue nuvole non vaghino a caso, ma tendano ad assomigliare ad una normale standard.
Questo  impedisce alla rete di collassare su soluzioni troppo rigide.
Continuità semantica significa che se prendo due punti vicini a queste nuvole, il decoder deve produrre due immagini molto simili, non ci sono più sbalzi improvvisi. Il passaggio da un gatto ad un cane, deve avvenire attraverso  una serie di passaggi intermedi.

Approfondiamo perchè questa distribuzione risolve i problemi della passato (AE)

Perchè le Distribuzioni?
La fluidità del manifold.
Sostituire punti fissi con distribuzioni forza il decoder a essere robusto: deve saper ricostruire l'input partendo da qualsiasi campione estratto dalla nuvola.
Questa tecnica elimina i 'buchi' nello spazio latente che rendevano l'autoencoder classico inadatto alla generazione di nuovi dati.
Nel VAE poichè ogni dato è una nuvola che sfuma verso le altre, i buchi spariscono.
Il decoder non impara a memoria i punti, impara a nagivare tra le probabilità

Tutto questo è bellissmo in teoria, ma scontrandoci con la pratica dell'addestramento troviamo un ostacolo insormontabile.

Reparameterization Trick
Il problema del campionamento non derivabile.
Per addestare una rete neurale usiamo la backpropagation che richiede che ogni operazione sia derivabile (richiede che il gradiente fluisca dell'output all'input). Tuttavia, l'operazione di campionamento casuale rompe questa catena, non è più derivabile.
Se estraiamo un numero a caso da una distribuzione, il grandiente non può passare attraverso quel lancio di dadi. 
Il Reparameterization Trick è l'ingegnosa soluzione matematica che permette di mantenere la stocasticità senza rinunciare alla backpropagation.

Spostare il Rumore
Isolare la sorgente stocastica.
Il trucco consiste nello spostare il rumore all'esterno del percorso di addestramento. Invece di campionare direttamente z, diciamo che z è uguale alla media pià la deviazione standard moltiplicata per un rumore esterno.
Questa soluzione ha implicazioni sulla stabilità del gradiente.

Come traduciamo tutto questo in poche righe di codice?

Dettagli Implementativi
Dalla teorica al tensore.
In PyTorch o Keras, implementiamo questo passaggio definendo una funzione lambda o un layer custom che esegue l'operazione elemento per elemento.
Questo piccolo spostamento algebrico è ciò che ha reso possibile l'esistenza dei modelli generativi variazionali moderni.

Inferenza Approssimata
Il calcolo esatto della probabilità dei dati richiederebbe di integrare su tutte le possibili configurazioni dello spazio latente (ricordiamo che lo spazio latente è continuo e multidimensionale), un compito computazionalmente impossibile.
Per questo passiamo all'inferenza variazionale
Smettiamo di cercare la risposta perfetta ma cerchiamo l approssimazione migliore. 
E' qui che il compito dell'encoder diventa quello di un indivino statistico.

Definiamo una famiglia di stribuzioni semplici, le gaussiane 'q'(parametrizzate da una rete neurale) e chiediamo all'encoder di trovare quella che somiglia di più al vero posteriore sconosciuto.

VAE e GAN sono due filosifie opposte.
Le GAN cercano di produrre immagini bellissime anche a costo di ignorare parte dei dati.
Le VAE cercano di dare una spiegazione ad ogni singolo dato del dataset.
Questo porta a modelli molto più stabili.
Il prezzo da pagare? le immagini sono spesso più sfuocate perchè la rete preferisce essere vagamente corretta su tutto, anzichè precisissima su pochi campioni.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    """
    Classe che implementa un Variational Autoencoder (VAE).
    A differenza di un Autoencoder classico, il VAE modella lo spazio latente 
    come una distribuzione di probabilità invece di un singolo punto vettoriale.
    """
    def __init__(self, input_dim=784, hidden_dim=400, latent_dim=20):
        super(VAE, self).__init__()
        
        # DEFINIZIONE DELL'ENCODER
        # Trasforma l'immagine di input in una rappresentazione intermedia (hidden_dim)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )
        
        # PARAMETRI DELLA DISTRIBUZIONE LATENTE
        # L'output dell'encoder viene utilizzato per calcolare i due parametri 
        # fondamentali della distribuzione normale nello spazio latente:
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)       # Vettore delle medie (mu)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)   # Logaritmo della varianza (log_var)

        # DEFINIZIONE DEL DECODER
        # Prende un campione dallo spazio latente e tenta di ricostruire l'immagine originale
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
            nn.Sigmoid() # Funzione di attivazione per riportare i valori dei pixel nel range [0, 1]
        )

    def reparameterize(self, mu, logvar):
        """
        IL TRUCCO DELLA RIPARAMETRIZZAZIONE (Reparameterization Trick)
        Necessario per permettere la backpropagation attraverso un nodo stocastico.
        Invece di campionare direttamente da N(mu, std), campioniamo epsilon da N(0, 1)
        e calcoliamo z = mu + epsilon * std.
        """
        # Calcolo della deviazione standard partendo dal logaritmo della varianza
        std = torch.exp(0.5 * logvar)
        
        # Campionamento di un rumore casuale epsilon con la stessa dimensione di std
        eps = torch.randn_like(std)
        
        # Trasformazione: z ora è un campione derivabile rispetto a mu e logvar
        return mu + eps * std

    def forward(self, x):
        """
        Flusso dei dati attraverso il modello:
        1. L'immagine viene compressa dall'encoder.
        2. Vengono estratti mu e logvar.
        3. Si genera un punto z nello spazio latente tramite reparameterize.
        4. Il decoder ricostruisce l'immagine a partire da z.
        """
        # Fase 1: Compressione (appiattiamo l'immagine se necessario a 784 pixel)
        h = self.encoder(x.view(-1, 784))
        
        # Fase 2: Estrazione dei parametri distribuzionali
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        
        # Fase 3: Campionamento nello spazio latente
        z = self.reparameterize(mu, logvar)
        
        # Fase 4: Ricostruzione finale
        # Restituiamo l'immagine ricostruita insieme a mu e logvar per il calcolo della loss
        return self.decoder(z), mu, logvar
